# Train Model 1 on Colab or Kaggle

This notebook trains the repository's `model_1` CNN + Bi-LSTM + CTC speech-to-text model on the [VIVOS Vietnamese Speech Corpus for ASR](https://www.kaggle.com/datasets/kynthesis/vivos-vietnamese-speech-corpus-for-asr).

The notebook converts VIVOS `prompts.txt` files into the CSV format expected by `model_1/train.py`:

```csv
audio_path,transcript
/path/to/audio_001.wav,xin chao cac ban
/path/to/audio_002.wav,hom nay chung ta hop ve ke hoach
```

Important: the current repo vocabulary in `model_1/text.py` only supports ASCII `a-z`, apostrophe, and space. This notebook strips Vietnamese accents and maps `đ` to `d` before training.

## 1. Runtime Setup

In Colab, set **Runtime > Change runtime type > GPU**. In Kaggle, enable **Accelerator > GPU** and attach your dataset from the notebook sidebar.

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = Path("/kaggle/input").exists()

print(f"Python: {sys.version}")
print(f"Colab: {IN_COLAB}")
print(f"Kaggle: {IN_KAGGLE}")
print(f"Working directory: {Path.cwd()}")

In [ ]:
# Install only if the runtime does not already include the needed packages.
try:
    import torch
    import torchaudio
    print("torch", torch.__version__)
    print("torchaudio", torchaudio.__version__)
except Exception:
    %pip install -q torch torchaudio
    import torch
    import torchaudio
    print("torch", torch.__version__)
    print("torchaudio", torchaudio.__version__)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Put the Repository in the Runtime

Choose one option:

- Upload this project folder or a project ZIP into Colab/Kaggle and set `PROJECT_DIR` to it.
- Or set `GITHUB_REPO_URL` and run the clone cell.

On Kaggle, the notebook working area is usually `/kaggle/working`; attached read-only datasets are under `/kaggle/input`.

In [ ]:
# Optional: clone the repository if it is hosted on GitHub.
# Example: GITHUB_REPO_URL = "https://github.com/your-user/your-repo.git"
GITHUB_REPO_URL = ""
PROJECT_DIR = Path.cwd()

if GITHUB_REPO_URL:
    repo_name = GITHUB_REPO_URL.rstrip("/").removesuffix(".git").split("/")[-1]
    target = Path("/kaggle/working" if IN_KAGGLE else "/content") / repo_name
    if not target.exists():
        !git clone {GITHUB_REPO_URL} {target}
    PROJECT_DIR = target

MODEL_DIR = PROJECT_DIR / "model_1"
assert MODEL_DIR.exists(), f"Could not find model_1 at {MODEL_DIR}. Set PROJECT_DIR to your repository root."
os.chdir(PROJECT_DIR)
print("Project directory:", PROJECT_DIR)
print("Model directory:", MODEL_DIR)

## 3. Prepare the VIVOS Dataset

On Kaggle, attach the dataset named `kynthesis/vivos-vietnamese-speech-corpus-for-asr`; Kaggle will mount it under `/kaggle/input`. On Colab, this cell tries to download the public Kaggle dataset with `kagglehub`.

In [ ]:
VIVOS_DATASET_SLUG = "kynthesis/vivos-vietnamese-speech-corpus-for-asr"
VIVOS_KAGGLE_DIR = Path("/kaggle/input/vivos-vietnamese-speech-corpus-for-asr")

def find_vivos_root(base):
    base = Path(base)
    if not base.exists():
        return None
    for train_prompts in base.rglob("train/prompts.txt"):
        root = train_prompts.parent.parent
        if (root / "test" / "prompts.txt").exists():
            return root
    return None

VIVOS_ROOT = find_vivos_root(VIVOS_KAGGLE_DIR)

if VIVOS_ROOT is None and IN_COLAB:
    try:
        import kagglehub
    except Exception:
        %pip install -q kagglehub
        import kagglehub
    downloaded_path = Path(kagglehub.dataset_download(VIVOS_DATASET_SLUG))
    VIVOS_ROOT = find_vivos_root(downloaded_path)

if VIVOS_ROOT is None:
    search_roots = [Path("/kaggle/input"), Path("/content"), Path.cwd()]
    for root in search_roots:
        VIVOS_ROOT = find_vivos_root(root)
        if VIVOS_ROOT is not None:
            break

assert VIVOS_ROOT is not None, "Could not find VIVOS train/test prompts.txt. Attach the Kaggle dataset or set VIVOS_ROOT manually."
print("VIVOS root:", VIVOS_ROOT)

In [ ]:
import csv
import re
import unicodedata

WORKING_DIR = Path("working/vivos")
TRAIN_CSV = WORKING_DIR / "train.csv"
VALID_CSV = WORKING_DIR / "valid.csv"

def normalize_for_current_vocab(text):
    text = text.lower().replace("\u0111", "d").replace("\u0110", "d")
    text = unicodedata.normalize("NFD", text)
    text = "".join(char for char in text if unicodedata.category(char) != "Mn")
    text = re.sub(r"[^a-z' ]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def load_vivos_prompts(split_dir):
    prompts_path = Path(split_dir) / "prompts.txt"
    rows = []
    with prompts_path.open(encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            utterance_id, transcript = line.split(maxsplit=1)
            rows.append((utterance_id, transcript))
    return rows

def build_audio_index(split_dir):
    return {path.stem: path for path in Path(split_dir).rglob("*.wav")}

def write_vivos_csv(split_name, output_csv):
    split_dir = VIVOS_ROOT / split_name
    prompts = load_vivos_prompts(split_dir)
    audio_by_id = build_audio_index(split_dir)
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    written = 0
    missing_audio = []
    with output_csv.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=["audio_path", "transcript", "original_transcript"])
        writer.writeheader()
        for utterance_id, original_transcript in prompts:
            audio_path = audio_by_id.get(utterance_id)
            transcript = normalize_for_current_vocab(original_transcript)
            if audio_path is None:
                missing_audio.append(utterance_id)
                continue
            if not transcript:
                continue
            writer.writerow({
                "audio_path": str(audio_path),
                "transcript": transcript,
                "original_transcript": original_transcript,
            })
            written += 1
    if missing_audio[:5]:
        print(f"Missing audio examples for {split_name}:", missing_audio[:5])
    print(f"{split_name}: wrote {written} rows to {output_csv}")
    return output_csv

TRAIN_CSV = write_vivos_csv("train", TRAIN_CSV)
VALID_CSV = write_vivos_csv("test", VALID_CSV)

def inspect_csv(csv_path, limit=5):
    csv_path = Path(csv_path)
    if not csv_path.exists():
        raise FileNotFoundError(f"Missing CSV: {csv_path}")
    with csv_path.open(newline="", encoding="utf-8") as handle:
        rows = list(csv.DictReader(handle))
    if not rows:
        raise ValueError(f"CSV has no rows: {csv_path}")
    missing = [col for col in ("audio_path", "transcript") if col not in rows[0]]
    if missing:
        raise ValueError(f"CSV is missing columns {missing}: {csv_path}")
    print(f"{csv_path}: {len(rows)} rows")
    for row in rows[:limit]:
        audio_path = Path(row["audio_path"])
        exists = audio_path.exists()
        print({"audio_path": row["audio_path"], "audio_exists": exists, "transcript": row["transcript"][:80]})
    return rows

train_rows = inspect_csv(TRAIN_CSV)
valid_rows = inspect_csv(VALID_CSV)

The generated CSV keeps `original_transcript` for inspection, but `model_1/train.py` only reads `audio_path` and the normalized `transcript`.

In [ ]:
print("Training CSV:", TRAIN_CSV.resolve())
print("Validation CSV:", VALID_CSV.resolve())
print("Original validation text example:", valid_rows[0].get("original_transcript"))
print("Normalized validation text example:", valid_rows[0].get("transcript"))

## 4. Train

Start with a small epoch count for a smoke test. Increase `EPOCHS` once the paths and audio loading are confirmed.

In [ ]:
EPOCHS = 30
BATCH_SIZE = 8
LEARNING_RATE = 3e-4
N_MFCC = 40
SAMPLE_RATE = 16000
CHECKPOINT = Path("artifacts/model1.pt")

CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
print("Checkpoint will be saved to:", CHECKPOINT.resolve())

In [ ]:
!python model_1/train.py \
  --train_csv "{TRAIN_CSV}" \
  --valid_csv "{VALID_CSV}" \
  --epochs {EPOCHS} \
  --batch_size {BATCH_SIZE} \
  --learning_rate {LEARNING_RATE} \
  --n_mfcc {N_MFCC} \
  --sample_rate {SAMPLE_RATE} \
  --checkpoint "{CHECKPOINT}"

## 5. Checkpoint and Inference Smoke Test

In [ ]:
checkpoint = torch.load(CHECKPOINT, map_location="cpu")
print("Saved checkpoint:", CHECKPOINT.resolve())
print("Best validation WER:", checkpoint.get("best_wer"))
print("Config:", checkpoint.get("config"))
print("Sample rate:", checkpoint.get("sample_rate"))

history = checkpoint.get("history", [])
if history:
    last_epoch = history[-1]
    print("Last saved epoch:", last_epoch["epoch"])
    print("Train metrics:", last_epoch["train"])
    print("Validation metrics:", last_epoch["valid"])

In [ ]:
# Run inference on the first validation audio file.
import json

sample_audio = valid_rows[0]["audio_path"]
payload = json.dumps({"audio_path": sample_audio, "checkpoint": str(CHECKPOINT)})
print("Audio:", sample_audio)
print("Reference:", valid_rows[0]["transcript"])
result = !echo '{payload}' | python model_1/infer.py
print("\n".join(result))

## 6. Download or Save the Model

Colab can download directly. Kaggle keeps output files in `/kaggle/working`, so the checkpoint appears in the notebook output after the run finishes.

In [ ]:
if IN_COLAB:
    from google.colab import files
    files.download(str(CHECKPOINT))
else:
    print("Checkpoint path:", CHECKPOINT.resolve())
    print("On Kaggle, open the Output panel after saving/running the notebook to download this file.")

## 7. Use the Checkpoint in the Web App

After copying `artifacts/model1.pt` back into the project, enable the Python inference bridge with:

```env
MODEL1_STT_ENABLED=true
MODEL1_PYTHON=python
MODEL1_SCRIPT=model_1/infer.py
MODEL1_CHECKPOINT=artifacts/model1.pt
```